# 4.5 DBSCAN, OPTICS, and HDBSCAN

DBSCAN, OPTICS & HDBSCAN turns unlabeled data into structure by choosing the right notion of similarity, compression, or surprise.

Distances and neighborhoods define dense regions separated by sparse regions. DBSCAN labels core, border, and noise points; OPTICS helps inspect density scale when one epsilon is too brittle.

Save a copy to Drive to edit. This notebook is deterministic, CPU-only, and uses only bundled scikit-learn data.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from collections import deque
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import AffinityPropagation
from sklearn.cluster import Birch
from sklearn.cluster import DBSCAN
from sklearn.cluster import KMeans
from sklearn.cluster import MeanShift
from sklearn.cluster import OPTICS
from sklearn.cluster import estimate_bandwidth
from sklearn.datasets import load_digits
from sklearn.datasets import load_iris
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import pairwise_distances
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

SEED = 7
rng = np.random.default_rng(SEED)

def cluster_ladder():
    """D1..D5 clustering ladder of rising difficulty. Returns [(name, X, y_true, k), ...].

    y_true is the generating label (for ARI scoring only — clustering does not see it).
    Rungs: hand points -> clean blobs -> anisotropic/overlap -> real Iris -> real digits(4-class).
    """
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.3, 0.2], [3.0, 3.0], [3.2, 2.8], [0.1, 3.1], [0.2, 2.9]])
    y1 = np.array([0, 0, 1, 1, 2, 2])
    rungs.append(("D1 hand 3 clusters", x1, y1, 3))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.7, random_state=1)
    rungs.append(("D2 clean blobs", x2, y2, 3))

    x3, y3 = make_blobs(n_samples=240, centers=3, cluster_std=1.6, random_state=2)
    transform = np.array([[0.6, -0.6], [-0.4, 0.8]])
    x3 = x3 @ transform
    rungs.append(("D3 anisotropic + overlap", x3, y3, 3))

    iris = load_iris()
    rungs.append(("D4 Iris (real, 4-D)", iris.data, iris.target, 3))

    digits = load_digits()
    keep = np.isin(digits.target, [0, 1, 2, 3])
    rungs.append(("D5 digits 0-3 (real, 64-D)", digits.data[keep] / 16.0, digits.target[keep], 4))

    return rungs



def project_2d(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 2:
        return X
    return PCA(n_components=2, random_state=SEED).fit_transform(X)


def ari_score(y_true, labels):
    return float(adjusted_rand_score(y_true, labels))


def safe_silhouette(X, labels):
    labels = np.asarray(labels)
    keep = labels != -1
    unique = np.unique(labels[keep])
    if keep.sum() < 3:
        return np.nan
    if unique.size < 2:
        return np.nan
    if unique.size >= keep.sum():
        return np.nan
    return float(silhouette_score(X[keep], labels[keep]))


def preview_ladder(rungs):
    rows = []
    for idx, item in enumerate(rungs, start=1):
        name, X, y_true, k = item
        rows.append((idx, name, X.shape, int(np.unique(y_true).size), k, np.round(X[:3], 3).tolist()))
    return rows


def plot_cluster_panels(results, title, show_centers=False):
    fig, axes = plt.subplots(1, len(results), figsize=(18, 3.4))
    for ax, result in zip(axes, results):
        Z = project_2d(result["X"])
        ax.scatter(Z[:, 0], Z[:, 1], c=result["labels"], s=16, cmap="tab10", alpha=0.82)
        if show_centers and result.get("centers") is not None:
            centers = np.asarray(result["centers"])
            if len(centers) > 0:
                centers_2d = centers if centers.shape[1] == 2 else PCA(n_components=2, random_state=SEED).fit(result["X"]).transform(centers)
                ax.scatter(centers_2d[:, 0], centers_2d[:, 1], c="black", marker="x", s=70)
        ax.set_title(f"D{result['rung']} ARI={result['ari']:.2f}")
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(title)
    plt.show()


def plot_ari_curve(results, title):
    xs = [result["rung"] for result in results]
    ys = [result["ari"] for result in results]
    plt.figure(figsize=(6, 3.5))
    plt.plot(xs, ys, marker="o")
    plt.ylim(-0.05, 1.05)
    plt.xlabel("D1 to D5 ladder rung")
    plt.ylabel("Adjusted Rand Index")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.show()


## 3. The concept, built once on D1

DBSCAN turns distance arithmetic into neighbor decisions. The lesson distances from $$J=\sum_i\min_k\lVert x_i-\mu_kVert_2^2$$ give $0.25$ for a nearby representative and $16.25$ for a far one, showing why $\epsilon$ scale matters.

In [ ]:
# Formula audit: $J = \sum_i \min_k ||x_i - \mu_k||_2^2$
x = np.array([1.0, 2.0])
mu1 = np.array([1.0, 1.5])
mu2 = np.array([4.5, 4.0])
d_close = float(np.sum((x - mu1) ** 2))
d_far = float(np.sum((x - mu2) ** 2))
J = 4 * d_close
assert d_close == 0.25
assert d_far == 16.25
assert J == 1.00
print(d_close, d_far, J)

Now implement real DBSCAN: identify core points, expand density-connected components, and leave sparse points as noise label -1. OPTICS is included later as the scale diagnostic; HDBSCAN is noted because it is not bundled with scikit-learn.

In [ ]:
def method(X, k=None, eps=0.8, min_samples=5):
    X = np.asarray(X, dtype=float)
    distances = pairwise_distances(X)
    neighbors = [np.where(row <= eps)[0].tolist() for row in distances]
    core = np.array([len(item) >= min_samples for item in neighbors])
    labels = np.full(len(X), -1, dtype=int)
    cluster_id = 0
    for start in range(len(X)):
        if labels[start] != -1 or not core[start]:
            continue
        labels[start] = cluster_id
        queue = deque(neighbors[start])
        while queue:
            point = queue.popleft()
            if labels[point] == -1:
                labels[point] = cluster_id
            if not core[point]:
                continue
            for candidate in neighbors[point]:
                if labels[candidate] == -1:
                    queue.append(candidate)
        cluster_id += 1
    centers = []
    for label in sorted(set(labels) - {-1}):
        centers.append(X[labels == label].mean(axis=0))
    if centers:
        centers = np.vstack(centers)
    else:
        centers = np.empty((0, X.shape[1]))
    return labels, centers, core


def choose_eps(X, min_samples=5):
    neighbors = NearestNeighbors(n_neighbors=min_samples)
    neighbors.fit(X)
    distances, indices = neighbors.kneighbors(X)
    return float(np.percentile(distances[:, -1], 75))

## 4. The dataset ladder

The same clustering function runs on D1 hand points, D2 clean blobs, D3 anisotropic overlap, D4 real Iris, and D5 real digits. Hidden labels are kept only for ARI scoring; the clustering method never receives them.

In [ ]:
rungs = cluster_ladder()
for row in preview_ladder(rungs):
    print(row)

## 5. Run the same method across D1-D5

The metric is Adjusted Rand Index against hidden generating labels. The labels are never passed into `method`; they are used only after clustering for evaluation.

In [ ]:
results = []
for rung, (name, X, y_true, k) in enumerate(rungs, start=1):
    X_use = StandardScaler().fit_transform(X)
    eps = choose_eps(X_use, min_samples=max(3, min(8, k * 2)))
    labels, centers, core = method(X_use, eps=eps, min_samples=max(3, min(8, k * 2)))
    score = ari_score(y_true, labels)
    sil = safe_silhouette(X_use, labels)
    noise_rate = float(np.mean(labels == -1))
    results.append({"rung": rung, "name": name, "X": X_use, "labels": labels, "centers": centers, "ari": score, "silhouette": sil, "noise_rate": noise_rate})
    print(f"D{rung} {name}: ARI={score:.3f} silhouette={sil:.3f} noise={noise_rate:.2f} eps={eps:.3f}")

## 6. Results visualization

Panels show density clusters and noise points. The curve tracks ARI while the table also reports noise rate.

In [ ]:
plot_cluster_panels(results, "DBSCAN cluster and noise assignments", show_centers=True)
plot_ari_curve(results, "DBSCAN ARI vs ladder rung")

## 7. Pitfall on D5: epsilon and min_samples sensitivity

Too-small $\epsilon$ can make digits all noise; too-large $\epsilon$ can collapse everything. We inspect k-distance values, compare failures, and use OPTICS as the bundled scale diagnostic. HDBSCAN is not used because it is not included in the standard scikit-learn bundle.

In [ ]:
name, X5, y5, k5 = rungs[-1]
X5s = StandardScaler().fit_transform(X5)
for eps in [0.2, choose_eps(X5s, min_samples=8), 10.0]:
    labels, centers, core = method(X5s, eps=eps, min_samples=8)
    print("DBSCAN eps", round(eps, 3), "clusters", len(set(labels) - {-1}), "noise", round(float(np.mean(labels == -1)), 3), "ARI", round(ari_score(y5, labels), 3))
optic = OPTICS(min_samples=8, xi=0.05)
optic_labels = optic.fit_predict(X5s)
print("OPTICS clusters", len(set(optic_labels) - {-1}), "noise", round(float(np.mean(optic_labels == -1)), 3), "ARI", round(ari_score(y5, optic_labels), 3))

## 8. Evaluate it + practice

- Metric: Adjusted Rand Index vs hidden labels, plus a no-skill sanity check from random labels.
- Sanity check: rerun with a nearby seed or hyperparameter and confirm the D5 story does not flip.
- Ablation: turn off the key idea in this lesson and watch ARI or stability drop.
- Failure signal: one cluster, all noise, or many singleton clusters usually means scale or hyperparameters dominate.
- D5 check: digits are real 64-D images, so visualization is a projection while clustering uses the full feature matrix.

Ablation: set `min_samples=1` and explain why many tiny clusters are not robust density structure.

Practice prompts:
1. Change one hyperparameter on D3 and explain whether ARI or the assignment panel changes first.

2. Add a stability rerun on D5 and compare the resulting ARI values.

3. Replace StandardScaler with no scaling on D4 or D5 and describe the failure mode.